In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("./star_dataset.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               1000 non-null   str    
 1   Distance (ly)      1000 non-null   float64
 2   Luminosity (L/Lo)  1000 non-null   float64
 3   Radius (R/Ro)      1000 non-null   float64
 4   Temperature (K)    1000 non-null   float64
 5   Spectral Class     1000 non-null   str    
dtypes: float64(4), str(2)
memory usage: 47.0 KB


In [6]:
df.describe()

,Distance (ly),Luminosity (L/Lo),Radius (R/Ro),Temperature (K)
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,295.505327,19644.909442,86.960696,9983.486779
std,541.478403,42223.595017,213.850005,7906.973529
min,3.877798,-4.993141,0.068087,2750.183163
25%,11.716853,10.441039,1.664479,3940.020856
50%,52.031435,171.097809,5.845444,7379.007975
75%,322.865874,10500.577117,33.719778,12055.975095
max,2600.490723,196004.854081,887.097936,28044.279272


In [7]:
print(df['Spectral Class'].value_counts())
print("\nMissing values:")
print(df.isnull().sum())


Spectral Class
A7V         74
A1V         73
A9II        48
B1III       45
M3.5V       45
M2Iab       44
G8III       39
M4Ve        38
A0V         38
K1.5III     38
B2III       37
M7IIIe      37
B0Ia        36
G2V         36
F7Ib        35
B1III-IV    32
A3V         31
B0.5IV      30
F5IV-V      30
B6Vep       29
M2.1V       27
B7V         26
M1.5Iab     26
A2Ia        25
K1V         24
M6V         22
K5III       18
B8Ia        17
Name: count, dtype: int64

Missing values:
Name                 0
Distance (ly)        0
Luminosity (L/Lo)    0
Radius (R/Ro)        0
Temperature (K)      0
Spectral Class       0
dtype: int64


In [8]:
# Identify issues: negative luminosity and duplicate star names
print("Negative luminosity rows:", (df['Luminosity (L/Lo)'] < 0).sum())
print("\nDuplicate star names:")
print(df[df.duplicated('Name', keep=False)].sort_values('Name')[['Name', 'Luminosity (L/Lo)', 'Temperature (K)']].head(10))


Negative luminosity rows: 88

Duplicate star names:
         Name  Luminosity (L/Lo)  Temperature (K)
420  Achernar        3148.316089     15040.775141
720  Achernar        3146.621300     15019.011484
721  Achernar        3150.941261     14980.023881
627  Achernar        3147.122215     15004.851530
726  Achernar        3153.991524     15023.277968
618  Achernar        3146.480171     14958.360315
479  Achernar        3154.323621     15030.422976
283  Achernar        3149.043659     15004.677455
753  Achernar        3151.110906     14985.780637
636  Achernar        3153.375674     15020.470410


In [9]:
# Remove negative luminosity rows
df_clean = df[df['Luminosity (L/Lo)'] >= 0].copy()
print(f"Rows before: {len(df)}  |  After removing negatives: {len(df_clean)}")


Rows before: 1000  |  After removing negatives: 912


In [10]:
# Handle duplicates: average numerical columns per star name, keep most common spectral class
numeric_cols = ['Distance (ly)', 'Luminosity (L/Lo)', 'Radius (R/Ro)', 'Temperature (K)']

df_numeric = df_clean.groupby('Name')[numeric_cols].mean()
df_spectral = df_clean.groupby('Name')['Spectral Class'].agg(lambda x: x.value_counts().index[0])

df_final = df_numeric.join(df_spectral).reset_index()
print(f"Unique stars after deduplication: {len(df_final)}")
df_final.head()


Unique stars after deduplication: 29


,Name,Distance (ly),Luminosity (L/Lo),Radius (R/Ro),Temperature (K),Spectral Class
0,Achernar,144.078936,3149.761447,9.209602,15003.610593,B6Vep
1,Acrux,320.916707,25000.665022,8.391405,28001.166630,B0.5IV
2,Aldebaran,65.028559,517.761826,44.179148,3923.614977,K5III
3,Alnilam,2000.090906,53699.999016,32.395467,27502.303666,B0Ia
4,Alpha Centauri B,4.360539,3.221930,0.885471,5260.399450,K1V


In [11]:
# NumPy operations on cleaned data
arr = df_final[numeric_cols].to_numpy()

print("Shape:", arr.shape)
print("Mean per column:", np.mean(arr, axis=0).round(2))
print("Std per column: ", np.std(arr, axis=0).round(2))

# Normalise to [0, 1] using broadcasting
col_range = arr.max(axis=0) - arr.min(axis=0)
arr_norm = (arr - arr.min(axis=0)) / np.where(col_range == 0, 1, col_range)
print("\nNormalised min:", arr_norm.min(axis=0).round(4))
print("Normalised max:", arr_norm.max(axis=0).round(4))


Shape: (29, 4)
Mean per column: [  324.53 21892.51    86.18  9955.88]
Std per column:  [  584.9  46098.66   205.84  7871.01]

Normalised min: [0. 0. 0. 0.]
Normalised max: [1. 1. 1. 1.]


In [12]:
# Export cleaned dataset for use in later phases
df_final.to_csv("./star_dataset_clean.csv", index=False)
print("Saved star_dataset_clean.csv")


Saved star_dataset_clean.csv
